# Load in data

In [1]:
import pandas as pd

def pretty_sample(df, col, n=20, random_state=42):
    print("Sampling from column:", col)
    samp_df = df[col].sample(n=n, random_state=random_state)
    for i, val in enumerate(samp_df):
        print(f"{i+1}. {val}")
        print("---")


n = pd.read_csv("../data/clean/normbank_predictions_3000.csv")
# Index(['setting', 'behavior', 'setting-behavior', 'constraints',
#        'constraints_given', 'constraint_predict', 'norm', 'label', 'split',
#        'idx'],

s = pd.read_csv("../data/human_stimuli_stratified_cleaned.csv")
# Index(['rot', 'action', 'situation', 'ssl_domain', 'agreement_condition'], dtype='object')

# Normbank

The way normbank is structured is that we have three core fields:
1. setting - where the behavior is taking place
2. behavior - what the person is doing
3. constraints - these are additional details like who the person doing the action is.

We turn this all into a string like "{behavior} at a {setting} where {constraints}".




## Clean normbank

We have already filtered to roles where the constraint is simple, but there is an awkward placeholder that says [PERSON] so there are different ways to handle this.

Let's also inspect the different norm labels: taboo, normal, expected.

In [2]:
norm_label_map = {
    0: "taboo",
    1: "normal",
    2: "expected"
}
n['norm_label'] = n['label'].map(norm_label_map)

In [9]:
def turn_into_row(x, replace_person=True):
    s = f"{x['behavior']} at a {x['setting']} where {x['constraints']}"
    if replace_person:
        s = s.replace("[PERSON]'s", "the person's")
        s = s.replace("[OTHER]'s", "the other person's")
    else:
        pass
    return s


norm_label_map = {
    0: "taboo",
    1: "normal",
    2: "expected"
}
n['norm_label'] = n['label'].map(norm_label_map)

n['stimulus_replace'] = n.apply(turn_into_row, axis=1)
n['stimulus_noreplace'] = n.apply(lambda x: turn_into_row(x, replace_person=False), axis=1)


print("RANDOM SAMPLES")
pretty_sample(n, 'stimulus_noreplace')
print("====================")
pretty_sample(n, 'stimulus_replace')


RANDOM SAMPLES
Sampling from column: stimulus_noreplace
1. boo the other player at a tennis court where [PERSON]'s role is 'referee'
---
2. baptize a child at a nursery where [PERSON]'s religion is not 'christianity'
---
3. scratch yourself at a elevator where [PERSON]'s role is 'a person who operates the elevator'
---
4. conduct a business meeting at a beach where temperature is 'freezing'
---
5. check for valid tickets at a auditorium where [PERSON]'s role is 'concession stand worker'
---
6. pick up a heavy object at a gift shop where [PERSON]'s role is 'store manager'
---
7. argue about the existence of a god at a locker room where [PERSON]'s religion is 'christianity'
---
8. eat a messy food at a subway platform where [PERSON]'s role is 'a person who is cleaning the station'
---
9. wear makeup at a art gallery where [PERSON]'s age bracket is 'gradeschooler'
---
10. have sex in the kitchen at a apartment where [PERSON]'s role is 'intruder'
---
11. write on a whiteboard at a classroo

## Explore by norm label

Normbank labels actions by ['taboo', 'normal', 'expected'], so let's sample some examples from each.

In [10]:
print("SAMPLE BY NORM LABEL")
for label in ['taboo', 'normal', 'expected']:
    print(f"START <--- NORM LABEL == '{label.upper()}' --->")
    pretty_sample(n[n['norm_label'] == label], 'stimulus_replace')
    print(f"END <--- NORM LABEL == '{label.upper()}' --->")
    print("====================")
    print("\n"*3)


SAMPLE BY NORM LABEL
START <--- NORM LABEL == 'TABOO' --->
Sampling from column: stimulus_replace
1. leave the water running at a apartment where the person's behavior is 'trying to flood the downstairs neighbors'
---
2. edit a story at a newsroom where the person's role is 'photojournalist'
---
3. hit on someone at a athletic field where the person's characteristic is 'creepy'
---
4. buy candy at a gas station where the person's behavior is 'try to pay with foreign currency'
---
5. go ice skating at a ski resort where the person's behavior is 'go outside without a coat'
---
6. wear pajamas at a airplane cabin where the person's role is 'air marshal'
---
7. wear a football jersey at a athletic field where the person's role is 'coach'
---
8. do a headstand at a dance studio where the person's age bracket is 'senior citizen'
---
9. have a house party at a apartment where the person's medical condition is 'head lice and nits'
---
10. talk to yourself at a bank where the person's behavior 

# Social Chemistry

Here's how social chemistry is structured. We have three fields:

1. action - what the person is doing
2. situation - the context in which the action is taking place
3. rot - the high level rule of thumb associated with the action-situation pair

Note: We did have a list of profanity words but I think we need some and can manually filter too. Crowdworkers sort of trolled the authors and a lot of these were explicit before the profanity filter.



## Action-Situation

It's originally called ['action', 'context'] but we renamed to ['action', 'situation'] because we thought that was clearer. The action is fairly specific but sometimes the context/situation is super vague (IMO). Maybe it makes to call it context?


In [5]:
def combine_action_situation(row):
    return f"Action: {row['action']}\nContext: {row['situation']}"

s['act_sit'] = s.apply(combine_action_situation, axis=1)
pretty_sample(s, 'act_sit')



Sampling from column: act_sit
1. Action: Supporting students.
Context: My chorus teacher said she had lost all respect for me.
---
2. Action: Roasting someone in a fun and lighthearted way, not hurtfully.
Context: Roasting my friend way too hard
---
3. Action: Not letting your girlfriend hang out with your friends if she does something that disrespects your friends.
Context: Not wanting my girlfriend to hang out with my friends
---
4. Action: Sharing personal feelings with strangers.
Context: Talking about my gender feelings
---
5. Action: Embarrassing a kid in good fun without taking it overboard.
Context: Daughter Cringes When Mom Does Happy Dance in Public
---
6. Action: Ignoring someone who likes you.
Context: Ghosting a man who's been into me for about two years
---
7. Action: Congratulating your competitor whenever they win something.
Context: Feeling terrible for not winning an award
---
8. Action: Going through an application process to adopt someone.
Context: Wanting to adopt 

## ROT

The other way we can use Social Chemistry is to ask people whether they agree with the rule of thumb (ROT). This can either be just rot alone or in conjunction with
the action-situation.

In [6]:
def combine_rot_action_situation(row):
    return f"ROT: {row['rot']}\nAction: {row['action']}\nContext: {row['situation']}"

s['rot_act_sit'] = s.apply(combine_rot_action_situation, axis=1)


pretty_sample(s, 'rot')
print("\n"*3)
pretty_sample(s, 'rot_act_sit')


Sampling from column: rot
1. Teachers are supposed to support students.
---
2. Roasting someone is meant to be fun and lighthearted, not hurtful.
---
3. It's fine to not let your girlfriend hang out with your friends if she does something that disrespects your friends.
---
4. It's best not to share personal feelings with strangers.
---
5. It is okay to embarrass a kid in good fun without taking it overboard.
---
6. You shouldn't ignore someone who likes you.
---
7. It is good to congratulate your competitor whenever they win something.
---
8. It's expected that you have to go through an application process to adopt someone.
---
9. It's wrong to vote against your partner.
---
10. It's understandable if you're suspicious when a widower in your family gets together with another widower soon after their spouses died.
---
11. It's okay if you are too sad to take on the responsibility of re-homing your pets.
---
12. It's acceptable to shun someone who hates people in your group.
---
13. It's

## Stratifying by agreement condition

In [11]:
for agreement in ['low_agreement', 'medium_agreement', 'high_agreement']:
    print(f"START <--- AGREEMENT == '{agreement.upper()}' --->")
    pretty_sample(s[s['agreement_condition'] == agreement], 'rot_act_sit', n=9)
    print(f"END <--- AGREEMENT == '{agreement.upper()}' --->")
    print("====================")
    print("\n"*3)

START <--- AGREEMENT == 'LOW_AGREEMENT' --->
Sampling from column: rot_act_sit
1. ROT: It is gross to sleep with your stepsister's best friend.
Action: Sleeping with your stepsister's best friend.
Context: Sleeping with my stepsister's friend
---
2. ROT: It's okay if you are too sad to take on the responsibility of re-homing your pets.
Action: Being too sad to take on the responsibility of re-homing your pets.
Context: Rehoming my wife's reptiles
---
3. ROT: You shouldn't ignore someone who likes you.
Action: Ignoring someone who likes you.
Context: Ghosting a man who's been into me for about two years
---
4. ROT: It is okay to embarrass a kid in good fun without taking it overboard.
Action: Embarrassing a kid in good fun without taking it overboard.
Context: Daughter Cringes When Mom Does Happy Dance in Public
---
5. ROT: It is loving to allow your wife to wear whatever she wants in the bedroom.
Action: Allowing your wife to wear whatever she wants in the bedroom.
Context: Wanting my 